# SafeVision AI — Clasificador niño/adulto

Entrena un clasificador YOLO11n-cls. Se ejecuta en Google Colab con GPU.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

**Nunca pegues tu clave en el cuaderno ni lo guardes en GitHub con ella.**

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Descargar el código del módulo IoT

In [ ]:
import os

if not os.path.exists("/content/SafeVisionAI-IoT"):
    !git clone https://github.com/Luisaparrac/SafeVisionAI-IoT.git /content/SafeVisionAI-IoT
%cd /content/SafeVisionAI-IoT
!pip install -q ultralytics pyyaml roboflow

## 3. Descargar el dataset

En Roboflow elige **Download Dataset → Folder Structure → Show download code**. En la celda de abajo cambia solo el workspace, el proyecto y la versión.

**Cómo crear el secreto en Colab:**

1. Abre el panel de secretos con el **ícono de llave**, a la izquierda.
2. Pulsa **Add new secret**.
3. En *Name* escribe `ROBOFLOW_API_KEY` y en *Value* pega tu clave.
4. **Activa el acceso** para este cuaderno.

**Nunca pegues tu clave en el cuaderno ni lo guardes en GitHub con ella.**

In [ ]:
# Replace the workspace, project and version with yours
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("TU_WORKSPACE").project("TU_PROYECTO")
version = project.version(1)
dataset = version.download("folder")

DATA_DIR = dataset.location

## 4. Entrenar

In [ ]:
!python training/train_child_model.py --data {DATA_DIR} --epochs 30 --device 0

## 5. Revisar los resultados

**top1_acc:** parte de las imágenes de validación clasificadas correctamente.

In [ ]:
import glob
import os
from IPython.display import Image, display

found = sorted(glob.glob("/content/**/child_model*/results.png", recursive=True), key=os.path.getmtime)
if not found:
    print("No se encontraron gráficas. Ejecuta primero el paso 4.")
else:
    folder = os.path.dirname(found[-1])
    print(f"Resultados en: {folder}")
    for name in ["results.png", "confusion_matrix.png"]:
        image = os.path.join(folder, name)
        if os.path.exists(image):
            display(Image(filename=image, width=900))

## 6. Descargar el modelo

Guarda `child_model.pt` en la carpeta `models/` y agrega al `.env`:

```
CHILD_MODEL=models/child_model.pt
```

In [ ]:
from google.colab import files
files.download("models/child_model.pt")